# Part A Q3: POS Tagging and Syntactic Analysis

In [1]:
import re
from pathlib import Path

import nltk
from nltk import CFG, ChartParser, RegexpTagger
from nltk.tokenize import word_tokenize
from textblob import TextBlob
from textblob.taggers import PatternTagger

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

In [2]:
DATA_FILE = Path("../data/Data_2.txt")

with open(DATA_FILE, "r", encoding="utf-8") as f:
    text = f.read()

# The same NLTK token list feeds every tagger, so any difference in the tags
# below comes from the tagger itself and not from a difference in tokenisation.
tokens = word_tokenize(text)

print(text)
print("Tokens:", tokens)
print("Token count:", len(tokens))

The big black dog barked at the white cat and chased away.
Tokens: ['The', 'big', 'black', 'dog', 'barked', 'at', 'the', 'white', 'cat', 'and', 'chased', 'away', '.']
Token count: 13


## 3.1 POS Tagging Methods

In [3]:
# Method 1 - NLTK: a pre-trained perceptron model that reads the whole sentence,
# so a word is tagged from its context rather than from its spelling alone.
nltk_tags = nltk.pos_tag(tokens)

# Method 2 - TextBlob: does its own tokenisation internally, so it is given the
# raw text rather than the token list above. Its default backend is reported
# below because that is what explains the result.
blob = TextBlob(text)
blob_tags = blob.tags

# TextBlob's other backend, PatternTagger, is a different rule/lexicon engine and
# is included so the comparison is not just the default backend twice over.
blob_pattern_tags = TextBlob(text, pos_tagger=PatternTagger()).tags

# Method 3 - Regular expression: hand-written rules tried in order, first match
# wins. The final ".*" is a catch-all default for anything no rule matched.
patterns = [
    (r"^[Tt]he$|^[Aa]n?$", "DT"),        # determiners, listed explicitly
    (r"^(at|in|on|to|by|with|from)$", "IN"),  # common prepositions
    (r"^(and|or|but)$", "CC"),           # coordinating conjunctions
    (r".*ing$", "VBG"),                  # gerunds, by suffix
    (r".*ed$", "VBD"),                   # past tense, by suffix
    (r".*es$", "VBZ"),                   # 3rd person singular, by suffix
    (r"^[^\w\s]+$", "."),               # punctuation
    (r".*", "NN"),                       # default: assume a noun
]
regex_tagger = RegexpTagger(patterns)
regex_tags = regex_tagger.tag(tokens)

print("Method 1 - NLTK pos_tag")
print(nltk_tags)

print("\nMethod 2 - TextBlob (default backend:",
      type(blob.pos_tagger).__name__ + ")")
print(blob_tags)

print("\nMethod 2b - TextBlob with PatternTagger backend")
print(blob_pattern_tags)

print("\nMethod 3 - Regular expression tagger")
print(regex_tags)

Method 1 - NLTK pos_tag
[('The', 'DT'), ('big', 'JJ'), ('black', 'JJ'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'JJ'), ('cat', 'NN'), ('and', 'CC'), ('chased', 'VBD'), ('away', 'RB'), ('.', '.')]

Method 2 - TextBlob (default backend: NLTKTagger)
[('The', 'DT'), ('big', 'JJ'), ('black', 'JJ'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'JJ'), ('cat', 'NN'), ('and', 'CC'), ('chased', 'VBD'), ('away', 'RB')]

Method 2b - TextBlob with PatternTagger backend
[('The', 'DT'), ('big', 'JJ'), ('black', 'JJ'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'JJ'), ('cat', 'NN'), ('and', 'CC'), ('chased', 'VBN'), ('away', 'RB')]

Method 3 - Regular expression tagger
[('The', 'DT'), ('big', 'NN'), ('black', 'NN'), ('dog', 'NN'), ('barked', 'VBD'), ('at', 'IN'), ('the', 'DT'), ('white', 'NN'), ('cat', 'NN'), ('and', 'CC'), ('chased', 'VBD'), ('away', 'NN'), ('.', '.')]


## 3.2 POS Tagger Comparison

In [4]:
# Neither TextBlob backend returns an entry for the full stop, so tags are looked
# up by word and shown as "-" where the tagger produced nothing.
blob_map = dict(blob_tags)
pattern_map = dict(blob_pattern_tags)

print(f"{'Token':10} {'NLTK':6} {'TextBlob':10} {'Pattern':9} {'Regex':6} Agreement")
print("-" * 62)
disagreements = []
for (word, nltk_tag), (_, regex_tag) in zip(nltk_tags, regex_tags):
    blob_tag = blob_map.get(word, "-")
    pattern_tag = pattern_map.get(word, "-")
    agree = nltk_tag == blob_tag == pattern_tag == regex_tag
    if not agree:
        disagreements.append((word, nltk_tag, blob_tag, regex_tag))
    print(f"{word:10} {nltk_tag:6} {blob_tag:10} {pattern_tag:9} {regex_tag:6} "
          f"{'yes' if agree else 'NO'}")

print("\nTokens tagged: ", len(nltk_tags))
print("TextBlob tags returned:", len(blob_tags))
print("Tokens where all taggers disagree:", len(disagreements))

# The default TextBlob backend wraps NLTK's own tagger, so identical output is the
# expected result rather than a coincidence; PatternTagger is the real comparison.
print("\nNLTK vs TextBlob default - identical:", nltk_tags[:-1] == blob_tags)
pattern_diffs = [(w, n, pattern_map.get(w, "-")) for w, n in nltk_tags
                 if pattern_map.get(w, "-") not in (n, "-")]
print("NLTK vs TextBlob PatternTagger differs on:", pattern_diffs)

# Counted separately because the regex tagger's errors are the substantive ones,
# whereas the TextBlob difference is only the dropped full stop.
regex_errors = [d for d in disagreements if d[1] != d[3]]
print("\nRegex tagger differs from NLTK on:")
for word, nltk_tag, _, regex_tag in regex_errors:
    print(f"  {word:8} NLTK={nltk_tag:4} regex={regex_tag}")

print("\nRegex tagger accuracy vs NLTK: "
      f"{(len(nltk_tags) - len(regex_errors)) / len(nltk_tags):.1%}")

Token      NLTK   TextBlob   Pattern   Regex  Agreement
--------------------------------------------------------------
The        DT     DT         DT        DT     yes
big        JJ     JJ         JJ        NN     NO
black      JJ     JJ         JJ        NN     NO
dog        NN     NN         NN        NN     yes
barked     VBD    VBD        VBD       VBD    yes
at         IN     IN         IN        IN     yes
the        DT     DT         DT        DT     yes
white      JJ     JJ         JJ        NN     NO
cat        NN     NN         NN        NN     yes
and        CC     CC         CC        CC     yes
chased     VBD    VBD        VBN       VBD    NO
away       RB     RB         RB        NN     NO
.          .      -          -         .      NO

Tokens tagged:  13
TextBlob tags returned: 12
Tokens where all taggers disagree: 6

NLTK vs TextBlob default - identical: True
NLTK vs TextBlob PatternTagger differs on: [('chased', 'VBD', 'VBN')]

Regex tagger differs from NLTK on:
  b

## 3.3 Parse Tree Generation

In [5]:
# A context-free grammar for the sentence. Two rules are deliberately left
# ambiguous so the parser returns every reading rather than just one:
#   S  -> NP VP | S CC VP   coordination at verb-phrase level or sentence level
#   VP -> V Adv | V Prt     "away" as an adverb or as a phrasal-verb particle
grammar = CFG.fromstring("""
S   -> NP VP | S CC VP
NP  -> Det Nom | Det Nom PP
Nom -> Adj Nom | N
VP  -> V PP | V Adv | V Prt | VP CC VP
PP  -> P NP
Det -> 'The' | 'the'
Adj -> 'big' | 'black' | 'white'
N   -> 'dog' | 'cat'
V   -> 'barked' | 'chased'
P   -> 'at'
CC  -> 'and'
Adv -> 'away'
Prt -> 'away'
""")

# Punctuation is dropped because the grammar has no production for it.
parse_tokens = [t for t in tokens if t.isalpha()]
print("Tokens sent to the parser:", parse_tokens)

parser = ChartParser(grammar)
trees = list(parser.parse(parse_tokens))
print("Possible parse trees found:", len(trees))

Tokens sent to the parser: ['The', 'big', 'black', 'dog', 'barked', 'at', 'the', 'white', 'cat', 'and', 'chased', 'away']
Possible parse trees found: 4


In [6]:
# pretty_print() is used instead of tree.draw(): draw() opens a separate Tkinter
# window, which does not render in a notebook or in the report.
for i, tree in enumerate(trees, 1):
    print(f"Parse tree {i}")
    tree.pretty_print()
    print()

Parse tree 1
                                  S                                           
                             _____|____________________________________        
                            S                               |          |      
          __________________|_________                      |          |       
         |                            VP                    |          |      
         |                   _________|___                  |          |       
         NP                 |             PP                |          |      
  _______|____              |      _______|____             |          |       
 |           Nom            |     |            NP           |          |      
 |    ________|____         |     |    ________|____        |          |       
 |   |            Nom       |     |   |            Nom      |          |      
 |   |         ____|___     |     |   |         ____|___    |          |       
 |   |        |       Nom   |    

In [7]:
# The same trees in bracketed notation, which is the compact form used in the report.
for i, tree in enumerate(trees, 1):
    print(f"Parse tree {i}")
    print(tree)
    print()

Parse tree 1
(S
  (S
    (NP (Det The) (Nom (Adj big) (Nom (Adj black) (Nom (N dog)))))
    (VP
      (V barked)
      (PP (P at) (NP (Det the) (Nom (Adj white) (Nom (N cat)))))))
  (CC and)
  (VP (V chased) (Prt away)))

Parse tree 2
(S
  (S
    (NP (Det The) (Nom (Adj big) (Nom (Adj black) (Nom (N dog)))))
    (VP
      (V barked)
      (PP (P at) (NP (Det the) (Nom (Adj white) (Nom (N cat)))))))
  (CC and)
  (VP (V chased) (Adv away)))

Parse tree 3
(S
  (NP (Det The) (Nom (Adj big) (Nom (Adj black) (Nom (N dog)))))
  (VP
    (VP
      (V barked)
      (PP (P at) (NP (Det the) (Nom (Adj white) (Nom (N cat))))))
    (CC and)
    (VP (V chased) (Prt away))))

Parse tree 4
(S
  (NP (Det The) (Nom (Adj big) (Nom (Adj black) (Nom (N dog)))))
  (VP
    (VP
      (V barked)
      (PP (P at) (NP (Det the) (Nom (Adj white) (Nom (N cat))))))
    (CC and)
    (VP (V chased) (Adv away))))

